# Cocos2d-x Fine-Tune + Evaluate

One notebook to rule them all. Pulls latest code from GitHub — edit `scripts/config.py` in the repo to change settings.

**Steps:** Install → Mount Drive → Clone/pull repo → Train (with resume) → Evaluate vs RAG → Report

**Requirements:** GPU runtime (T4+). Runtime → Change runtime type → GPU.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────
!pip install -q transformers datasets peft accelerate bitsandbytes trl \
    rouge-score nltk sentence-transformers matplotlib pandas

In [ ]:
# ── 2. Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. Clone or pull latest from GitHub ──────────────────────────────
import os
REPO = 'nmnhut-it/fine-tune-cocoos'
REPO_DIR = '/content/fine-tune-cocoos'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/{REPO}.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ── 4. Init config ───────────────────────────────────────────────────
from scripts.config import ensure_drive_dirs
ensure_drive_dirs()
print('Drive directories ready.')

---
## Phase 1: Training
Run cells below to fine-tune. Checkpoints auto-save to Drive — re-run after a disconnect and it resumes.

In [ ]:
# ── 5. Load data + model ─────────────────────────────────────────────
from scripts.train import load_data, load_model_and_tokenizer, tokenize_datasets

train_ds, test_ds = load_data()
model, tokenizer = load_model_and_tokenizer()
train_tok, test_tok = tokenize_datasets(train_ds, test_ds, tokenizer)

In [ ]:
# ── 6. Train (auto-resumes from checkpoint) ─────────────────────────
from scripts.train import run_training

trainer = run_training(model, tokenizer, train_tok, test_tok)

In [ ]:
# ── 7. Save adapter + evaluate loss ─────────────────────────────────
from scripts.train import save_adapter, evaluate_loss

save_adapter(model, tokenizer)
evaluate_loss(trainer)

In [ ]:
# ── 8. Quick inference test ───────────────────────────────────────────
import torch
from scripts.config import ALPACA_PROMPT, MAX_NEW_TOKENS, TEMPERATURE, TOP_P

def quick_generate(instruction):
    prompt = ALPACA_PROMPT.format(instruction=instruction)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                             temperature=TEMPERATURE, top_p=TOP_P, do_sample=True)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

for i in [0, 25, 50, 75]:
    ex = test_ds[i]
    print(f"\n{'='*60}")
    print(f"Q: {ex['instruction'][:200]}")
    print(f"\nGenerated:\n{quick_generate(ex['instruction'])[:400]}")
    print(f"\nExpected:\n{ex['output'][:300]}")

---
## Phase 2: Evaluation — Fine-Tuned vs RAG (Context7 docs)
Compares the fine-tuned model against a RAG baseline that retrieves from the same source docs.
Also resumes if interrupted.

In [ ]:
# ── 9. Build RAG index from local docs ───────────────────────────────
# Free training model memory first
import gc, torch
del model, trainer, train_tok, test_tok
gc.collect()
torch.cuda.empty_cache()

from scripts.evaluate import build_rag_index
doc_chunks, doc_sources, embedder, chunk_embs = build_rag_index()

In [ ]:
# ── 10. Load base model + fine-tuned adapter ────────────────────────
from scripts.evaluate import load_models
base_model, ft_model, tokenizer = load_models()

In [ ]:
# ── 11. Run evaluation (resumes from partial results) ────────────────
from scripts.evaluate import run_evaluation

results = run_evaluation(
    base_model, ft_model, tokenizer,
    embedder, chunk_embs, doc_chunks, doc_sources,
)

In [ ]:
# ── 12. Metrics + summary ────────────────────────────────────────────
from scripts.evaluate import (
    compute_all_metrics, print_summary,
    build_comparison_df, plot_chart, save_report, show_side_by_side,
)

ft_metrics, rag_metrics = compute_all_metrics(results)
print_summary(ft_metrics, 'FINE-TUNED MODEL')
print_summary(rag_metrics, 'RAG (Context7 Docs)')

In [ ]:
# ── 13. Comparison table + chart ──────────────────────────────────────
df = build_comparison_df(results, ft_metrics, rag_metrics)
plot_chart(ft_metrics, rag_metrics)
df.head(20)

In [ ]:
# ── 14. Side-by-side examples ────────────────────────────────────────
show_side_by_side(results, ft_metrics, rag_metrics, df)

In [ ]:
# ── 15. Save report to Drive ─────────────────────────────────────────
save_report(results, ft_metrics, rag_metrics, df)

---
## (Optional) Push adapter to Hugging Face Hub

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# ft_model.push_to_hub('your-username/cocos2dx-coder-lora')
# tokenizer.push_to_hub('your-username/cocos2dx-coder-lora')